# Which LLM agrees with *my* grading style? (Eval Protocol)

You have a 0–3 correctness rubric and a set of cases you've already hand-scored (the gold labels). This notebook asks: **which candidate model, used as a rubric grader for my agent, best reproduces my scores?**

It scores each candidate model against your gold labels with two metrics:
- **Exact accuracy** — fraction of cases where the model's 0–3 score equals yours.
- **Quadratic-weighted Cohen's kappa** — chance-corrected agreement on an ordinal scale (1.0 = perfect, 0 = chance, <0 = worse than chance). This is the right "do two graders agree" statistic and it punishes being-off-by-2 more than off-by-1.

Generation runs through Eval Protocol's `SingleTurnRolloutProcessor` (the same rollout path as the Fireworks benchmark notebooks), so candidate graders across **both Fireworks and Anthropic** run through one uniform code path.

**Prereqs:** Jupyter kernel = conda `cookbook` env. `FIREWORKS_API_KEY` in `training/.env`, and `ANTHROPIC_API_KEY` in your shell (only if you keep Anthropic models in the list). Run the install cell once if imports fail.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q -e "../../.[eval]"

In [2]:
# --- edit these ---
# Candidate grader models. These are the IDs reachable on this account as of testing.
# A 404 / NotFoundError just means that model isn't deployed for your account -- swap in your own.
CANDIDATE_MODELS = [
    # Fireworks (needs FIREWORKS_API_KEY in training/.env) -- serverless on this account
    "fireworks_ai/accounts/fireworks/models/glm-5p1",
    "fireworks_ai/accounts/fireworks/models/minimax-m3",
    # Anthropic (needs ANTHROPIC_API_KEY in your shell)
    "anthropic/claude-haiku-4-5",
    "anthropic/claude-sonnet-4-5",
    "anthropic/claude-opus-4-8",
]

USE_SYNTHETIC = True   # True -> grade your 12 cases + a synthetic expansion (more power for kappa)
TEMPERATURE = 0.0
CONCURRENCY = 4        # concurrent grading calls per model

In [3]:
import asyncio
import json
import os
import re
from pathlib import Path

import litellm
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from eval_protocol.models import EvaluateResult, EvaluationRow
from eval_protocol.pytest import SingleTurnRolloutProcessor
from eval_protocol.pytest.types import RolloutProcessorConfig

# Fireworks key lives in training/.env; Anthropic key is expected in the shell env.
training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")

needs_anthropic = any(m.startswith("anthropic/") for m in CANDIDATE_MODELS)
needs_fireworks = any(m.startswith("fireworks_ai/") for m in CANDIDATE_MODELS)
if needs_anthropic and not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError("Set ANTHROPIC_API_KEY in your shell environment.")
if needs_fireworks and not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")

# drop_params=True: providers that don't support response_format just ignore it
# (keeps the same code path working across Fireworks + Anthropic).
litellm.drop_params = True

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TaskDefinitionModel(BaseModel):


## 1. The rubric and structured output schema

This is your 0–3 rubric, ported verbatim from `6_1_structured_grader_evaluation.ipynb`. Every candidate model is given the *same* rubric — we're measuring which model best matches your judgment, holding the instructions fixed.

In [4]:
class ScoreResponse(BaseModel):
    reasoning: str = Field(
        description="The reasoning process of what score you should pick. This should be detailed. "
        "Outline the points in the ground truth that the AI response correctly answered and incorrectly missed."
    )
    score: int = Field(description="An integer between 0 and 3 representing the correctness of the AI response compared to the ground truth")


SYSTEM_RUBRIC = (
    "You are an expert evaluator. "
    "Given an AI's response to a question and the ground truth answer, "
    "score the AI's response on a scale from 0 to 3 based on correctness:\n"
    "0 = Completely does not match the ground truth or is irrelevant\n"
    "1 = Partially matches the ground truth, but with major errors or omissions\n"
    "2 = Mostly matches the ground truth, but with at most a single minor error or at most a single missing detail\n"
    "3 = Completely matches the ground truth exactly\n\n"
    "Here are examples of each score with reasoning:\n\n"
    "EXAMPLE - Score 3 (Perfect Match):\n"
    "Question: What is the square root of 144?\n"
    "Ground Truth: 12\n"
    "AI Response: The square root of 144 is 12.\n"
    "Reasoning: The AI response correctly identifies the answer as 12, which matches the ground truth exactly. The additional context ('The square root of 144 is') does not detract from the correctness of the answer.\n"
    "Score: 3\n\n"
    "EXAMPLE - Score 2 (Minor Error):\n"
    "Question: Name the four seasons in order.\n"
    "Ground Truth: Spring, Summer, Fall, Winter\n"
    "AI Response: Spring, Summer, Autumn, Winter\n"
    "Reasoning: The AI response correctly identifies all four seasons in the correct order. However, it uses 'Autumn' instead of 'Fall'. Since 'Autumn' and 'Fall' are synonymous terms for the same season, this is a minor naming variation rather than a factual error. All other elements match the ground truth exactly.\n"
    "Score: 2\n\n"
    "EXAMPLE - Score 1 (Major Errors/Omissions):\n"
    "Question: List the planets in our solar system.\n"
    "Ground Truth: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune\n"
    "AI Response: Earth, Mars, Jupiter, Saturn\n"
    "Reasoning: The AI response correctly identifies 4 planets (Earth, Mars, Jupiter, Saturn), which are all included in the ground truth. However, it is missing 4 other planets (Mercury, Venus, Uranus, Neptune), representing a 50% omission rate. This is a major omission as half of the required information is missing. The planets provided are correct but incomplete.\n"
    "Score: 1\n\n"
    "EXAMPLE - Score 0 (Completely Wrong):\n"
    "Question: What is the chemical formula for water?\n"
    "Ground Truth: H2O\n"
    "AI Response: CO2\n"
    "Reasoning: The AI response provides CO2 (carbon dioxide) when the ground truth is H2O (water). These are completely different chemical compounds with different properties and compositions. The response does not match the ground truth in any way and provides factually incorrect information.\n"
    "Score: 0\n\n"
    'Respond with ONLY a JSON object of the form {"reasoning": "<your reasoning>", "score": <integer 0-3>}.'
)


def human_prompt(question: str, ground_truth: str, ai_response: str) -> str:
    return (
        f"Question: {question}\n"
        f"Ground Truth: {ground_truth}\n\n"
        f"AI Response: {ai_response}\n"
        "The goal is to match the AI response to the ground truth. Your score should be based on how well "
        "the AI response matches the ground truth, nothing else. Score the AI response from 0 to 3."
    )

## 2. The labeled dataset (your gold scores)

Your 12 hand-scored cases. With `USE_SYNTHETIC = True` we append a synthetic expansion that deliberately covers all four score bands — 12 cases is too few for a stable kappa, so the extra rows give the agreement metric more to work with. Edit / extend `GOLD_CASES` and `SYNTHETIC_CASES` to encode more of *your* grading style.

In [5]:
# Your original 12 hand-scored cases.
GOLD_CASES = [
    {"question": "For the driver who set the fastest lap speed, what is his nationality?", "ai_response": "This person is from Spain", "ground_truth": "Spanish", "expected_score": 3},
    {"question": "Which constructor has the highest point?", "ai_response": "The constructor with the highest points is Mercedes, with a total of 765 points.", "ground_truth": "[('Mercedes',)]", "expected_score": 3},
    {"question": "For all patients who are older than 55 years old, what is the percentage of female who has abnormal prothrombin time (PT)?", "ai_response": "For patients older than 55 years old who are female (SEX = 'F'), there are 8,744 total females and 153 of them have an abnormal prothrombin time (PT > 13.5).\n\nTo calculate the percentage of females with abnormal PT: (153 / 8744) * 100 = 1.75%\n\nSo, approximately 1.75% of females older than 55 years have an abnormal prothrombin time (PT).", "ground_truth": "[(1.2037369744879627,)]", "expected_score": 1},
    {"question": "What are the three primary colors and how do they combine to form other colors?", "ai_response": "The three primary colors are red, blue, and yellow. They combine to form other colors through color mixing.", "ground_truth": "The three primary colors are red, blue, and yellow. For example, red and blue make purple.", "expected_score": 2},
    {"question": "What is the total revenue for the year 2023 across all regions?", "ai_response": "The total revenue for 2023 was $4.2M, with North America contributing $2.1M, Europe $1.5M, and Asia $0.6M.", "ground_truth": "4200000", "expected_score": 3},
    {"question": "Which country won the most gold medals in the 2024 Olympics?", "ai_response": "The United States won the most gold medals with 40, followed by China with 40, but the US had more total medals.", "ground_truth": "USA and China (tie at 40)", "expected_score": 3},
    {"question": "What is the treatment for a common cold?", "ai_response": "Antibiotics are the primary treatment for a common cold to kill the virus.", "ground_truth": "Rest, fluids, and over-the-counter medications. Antibiotics do not work on viruses.", "expected_score": 0},
    {"question": "How many employees does the company have as of Q4 2023?", "ai_response": "As of the end of 2023, the company headcount was approximately 12,500.", "ground_truth": "12482", "expected_score": 2},
    {"question": "Calculate the derivative of f(x) = x^2 + 3x.", "ai_response": "f'(x) = 2x + 3x", "ground_truth": "2x + 3", "expected_score": 1},
    {"question": "What is the boiling point of water at sea level?", "ai_response": "Water boils at 100 degrees.", "ground_truth": "100 degrees Celsius or 212 degrees Fahrenheit", "expected_score": 2},
    {"question": "Who wrote 'To Kill a Mockingbird'?", "ai_response": "The novel was written by Harper Lee in 1960.", "ground_truth": "Harper Lee", "expected_score": 3},
    {"question": "For all patients who are older than 55 years old, what is the percentage of female who has abnormal prothrombin time (PT)?", "ai_response": "The percentage is 1.75%.", "ground_truth": "[(1.2037369744879627,)]", "expected_score": 0},
]

# Synthetic expansion, written to mirror the same grading style across all four bands.
SYNTHETIC_CASES = [
    # 3 = exact / semantically exact
    {"question": "What is the capital of France?", "ai_response": "The capital of France is Paris.", "ground_truth": "Paris", "expected_score": 3},
    {"question": "What is 15 multiplied by 4?", "ai_response": "15 x 4 = 60", "ground_truth": "60", "expected_score": 3},
    {"question": "In what year did World War II end?", "ai_response": "World War II ended in 1945.", "ground_truth": "1945", "expected_score": 3},
    # 2 = one minor error / one missing detail
    {"question": "List the primary states of matter.", "ai_response": "Solid, liquid, and gas.", "ground_truth": "Solid, liquid, gas, and plasma.", "expected_score": 2},
    {"question": "What is the freezing point of water?", "ai_response": "Water freezes at 0 degrees.", "ground_truth": "0 degrees Celsius (32 degrees Fahrenheit)", "expected_score": 2},
    {"question": "Name the author and publication year of '1984'.", "ai_response": "It was written by George Orwell.", "ground_truth": "George Orwell, published in 1949", "expected_score": 2},
    # 1 = major errors / large omissions
    {"question": "List the first four prime numbers.", "ai_response": "2 and 3.", "ground_truth": "2, 3, 5, 7", "expected_score": 1},
    {"question": "What is the area of a circle with radius 2? (use pi=3.14)", "ai_response": "The area is about 12.56, computed as 2 * pi * r.", "ground_truth": "12.56 (pi * r^2)", "expected_score": 1},
    {"question": "Which planets are gas giants?", "ai_response": "Jupiter.", "ground_truth": "Jupiter, Saturn, Uranus, Neptune", "expected_score": 1},
    # 0 = wrong / irrelevant
    {"question": "What is the chemical symbol for gold?", "ai_response": "The chemical symbol for gold is Gd.", "ground_truth": "Au", "expected_score": 0},
    {"question": "Who painted the Mona Lisa?", "ai_response": "It was painted by Vincent van Gogh.", "ground_truth": "Leonardo da Vinci", "expected_score": 0},
    {"question": "What is the speed of light in a vacuum?", "ai_response": "I prefer not to discuss physics.", "ground_truth": "Approximately 299,792,458 meters per second", "expected_score": 0},
]

CASES = GOLD_CASES + (SYNTHETIC_CASES if USE_SYNTHETIC else [])
print(f"{len(CASES)} labeled cases ({len(GOLD_CASES)} gold + {len(SYNTHETIC_CASES) if USE_SYNTHETIC else 0} synthetic)")
from collections import Counter
print("Label distribution:", dict(sorted(Counter(c['expected_score'] for c in CASES).items())))

24 labeled cases (12 gold + 12 synthetic)
Label distribution: {0: 5, 1: 5, 2: 6, 3: 8}


## 3. Grade every case with every candidate model (via `SingleTurnRolloutProcessor`)

Each case becomes an `EvaluationRow` whose messages are `[rubric (system), case (user)]` and whose `ground_truth` is your gold score. We run Eval Protocol's `SingleTurnRolloutProcessor` once per candidate model — that's the same rollout path the Fireworks benchmark notebooks use — then parse the integer score out of the trailing assistant message. The grader records exact-match in `evaluation_result`; the raw predicted score is kept for the kappa computation.

In [6]:
_SCORE_RE = re.compile(r'"?score"?\s*[:=]\s*(-?\d+)', re.IGNORECASE)


def model_response(row: EvaluationRow) -> str:
    return str(row.messages[-1].content) if row.messages else ""


def parse_score(content: str) -> int | None:
    """Pull the integer score out of a structured/JSON-ish grader response."""
    if not content:
        return None
    try:
        return int(ScoreResponse.model_validate_json(content).score)
    except Exception:
        pass
    try:
        return int(json.loads(content).get("score"))
    except Exception:
        pass
    m = _SCORE_RE.search(content)
    return int(m.group(1)) if m else None


def grade_agreement(row: EvaluationRow) -> EvaluateResult:
    """Exact-match against your gold score; predicted score is stashed in reason for later metrics."""
    pred = parse_score(model_response(row))
    gt = int(row.ground_truth)
    if pred is None:
        return EvaluateResult(score=0.0, reason=f"pred=None gt={gt}")
    return EvaluateResult(score=1.0 if pred == gt else 0.0, reason=f"pred={pred} gt={gt}")


def build_rows() -> list[EvaluationRow]:
    rows = []
    for i, c in enumerate(CASES):
        row = EvaluationRow(
            messages=[
                {"role": "system", "content": SYSTEM_RUBRIC},
                {"role": "user", "content": human_prompt(c["question"], c["ground_truth"], c["ai_response"])},
            ],
            ground_truth=str(c["expected_score"]),
        )
        row.input_metadata.row_id = f"case-{i}"
        rows.append(row)
    return rows


# NOTE: we do NOT pass response_format here. The processor serializes completion_params,
# so a Pydantic *class* raises PydanticSerializationError, and {"type":"json_object"} made
# some models emit unparseable output. The rubric already mandates a JSON object, and
# parse_score recovers the integer robustly across both providers.
async def grade_with_model(model: str) -> tuple[list[EvaluationRow], list[str]]:
    processor = SingleTurnRolloutProcessor(drop_trailing_assistant_messages=True)
    config = RolloutProcessorConfig(
        completion_params={"model": model, "temperature": TEMPERATURE},
        mcp_config_path="",  # MCP config is for agent/tool-calling evals
        semaphore=asyncio.Semaphore(CONCURRENCY),
    )
    results = await asyncio.gather(*processor(build_rows(), config), return_exceptions=True)
    ev_rows = [r for r in results if isinstance(r, EvaluationRow)]
    errors = [f"{type(r).__name__}: {r}" for r in results if isinstance(r, Exception)]
    for r in ev_rows:
        r.evaluation_result = grade_agreement(r)
    return ev_rows, errors


predictions = {}  # model -> completed EvaluationRows (each with evaluation_result set)
for model in CANDIDATE_MODELS:
    print(f"Grading with {model} ...", end=" ", flush=True)
    rows_done, errors = await grade_with_model(model)
    predictions[model] = rows_done
    n_parsed = sum(1 for r in rows_done if parse_score(model_response(r)) is not None)
    print(f"done ({len(rows_done)}/{len(CASES)} rolled out, {n_parsed} scores parsed, {len(errors)} errors)")
    if errors:
        print(f"    first error: {errors[0][:160]}")

Grading with fireworks_ai/accounts/fireworks/models/glm-5p1 ... done (24/24 rolled out, 24 scores parsed, 0 errors)
Grading with fireworks_ai/accounts/fireworks/models/minimax-m3 ... done (24/24 rolled out, 24 scores parsed, 0 errors)
Grading with anthropic/claude-haiku-4-5 ... done (24/24 rolled out, 24 scores parsed, 0 errors)
Grading with anthropic/claude-sonnet-4-5 ... done (24/24 rolled out, 24 scores parsed, 0 errors)
Grading with anthropic/claude-opus-4-8 ... 
Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/n

## 4. Agreement metrics

`exact accuracy` = fraction of exact 0–3 matches with your labels. `quadratic kappa` = quadratic-weighted Cohen's kappa vs your labels (implemented inline, no sklearn). Cases the model failed to score (parse/API error) are dropped from that model's metrics and reported as `n_scored`.

In [7]:
LABELS = [0, 1, 2, 3]
VALID = set(LABELS)


def quadratic_kappa(y_true: list[int], y_pred: list[int]) -> float:
    """Quadratic-weighted Cohen's kappa over the fixed 0..3 ordinal scale."""
    n = len(y_true)
    if n == 0:
        return float("nan")
    k = len(LABELS)
    idx = {v: i for i, v in enumerate(LABELS)}
    O = [[0.0] * k for _ in range(k)]
    for t, p in zip(y_true, y_pred):
        O[idx[t]][idx[p]] += 1.0
    row = [sum(O[i]) for i in range(k)]
    col = [sum(O[i][j] for i in range(k)) for j in range(k)]
    W = [[((i - j) ** 2) / ((k - 1) ** 2) for j in range(k)] for i in range(k)]
    E = [[row[i] * col[j] / n for j in range(k)] for i in range(k)]
    num = sum(W[i][j] * O[i][j] for i in range(k) for j in range(k))
    den = sum(W[i][j] * E[i][j] for i in range(k) for j in range(k))
    return 1.0 - num / den if den else float("nan")


rows = []
for model in CANDIDATE_MODELS:
    pairs = []
    for r in predictions[model]:
        pred = parse_score(model_response(r))
        gt = int(r.ground_truth)
        if pred in VALID:
            pairs.append((gt, pred))
    y_true = [t for t, p in pairs]
    y_pred = [p for t, p in pairs]
    n = len(y_true)
    acc = sum(t == p for t, p in pairs) / n if n else float("nan")
    mae = sum(abs(t - p) for t, p in pairs) / n if n else float("nan")
    kappa = quadratic_kappa(y_true, y_pred)
    rows.append({"model": model, "n_scored": n, "exact_acc": acc, "mae": mae, "quad_kappa": kappa})

rows.sort(key=lambda r: (-(r["quad_kappa"] if r["quad_kappa"] == r["quad_kappa"] else -9), -r["exact_acc"]))

print(f"Agreement with your gold labels over {len(CASES)} cases\n")
print(f"{'model':<55} {'n':>4} {'exact':>8} {'MAE':>7} {'q-kappa':>9}")
print("-" * 86)
for r in rows:
    print(f"{r['model']:<55} {r['n_scored']:>4} {r['exact_acc']:>7.1%} {r['mae']:>7.2f} {r['quad_kappa']:>9.3f}")
print("\nBest grader for your style:", rows[0]["model"] if rows else "n/a")

Agreement with your gold labels over 24 cases

model                                                      n    exact     MAE   q-kappa
--------------------------------------------------------------------------------------
anthropic/claude-sonnet-4-5                               24   87.5%    0.12     0.950
anthropic/claude-haiku-4-5                                24   79.2%    0.21     0.923
fireworks_ai/accounts/fireworks/models/glm-5p1            24   79.2%    0.21     0.918
fireworks_ai/accounts/fireworks/models/minimax-m3         24   75.0%    0.29     0.861
anthropic/claude-opus-4-8                                  0    nan%     nan       nan

Best grader for your style: anthropic/claude-sonnet-4-5


## 5. Where do the models disagree with you?

Per-case breakdown for the top model — inspect the rows where it diverged from your label to sanity-check whether *it* is wrong or your label is.

In [8]:
top_model = rows[0]["model"] if rows else CANDIDATE_MODELS[0]
print(f"Disagreements for {top_model}:\n")
any_diff = False
for r in predictions[top_model]:
    pred = parse_score(model_response(r))
    gt = int(r.ground_truth)
    if pred != gt:
        any_diff = True
        user_text = next((str(m.content) for m in r.messages if m.role == "user"), "")
        q = user_text.split("Ground Truth:")[0].replace("Question:", "").strip()[:70].replace("\n", " ")
        print(f"  you={gt} model={pred}  | {q}...")
if not any_diff:
    print("  none — perfect agreement on every case.")

Disagreements for anthropic/claude-sonnet-4-5:

  you=0 model=1  | For all patients who are older than 55 years old, what is the percenta...
  you=2 model=1  | List the primary states of matter....
  you=2 model=1  | Name the author and publication year of '1984'....


## Next steps

- Expand `GOLD_CASES` with real examples from *your* agent's domain — the more of your own hand-scored cases, the more trustworthy the kappa.
- Pick the top model as your agent's rubric grader. If two are close on kappa, prefer the cheaper/faster one (e.g. a Fireworks open model over a frontier Anthropic model).
- To harden against label noise, score each case 3x at `TEMPERATURE>0` and check per-model self-consistency before trusting agreement.